# Use Cases Of An AI Engineer

Run `4-classify-job-postings.ipynb` first so that `jobs/2-classified-jobs.jsonl` exists.

This notebook reads the LLM-filtered AI engineer job postings and uses `gpt-5.4-mini` to summarize the common AI application use cases these companies are building.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display


In [ ]:

load_dotenv(override=True)

### Load classified jobs from file

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

if classified_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The classified jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

jobs_for_summary = pd.read_json(classified_jobs_path, lines=True)

# Keep only the jobs the LLM classified as true AI engineering roles
jobs_for_summary = jobs_for_summary[jobs_for_summary["is_ai_engineering_role"]]

print(f"Loaded {len(jobs_for_summary)} classified AI engineering job postings")

### Merge all descriptions into a single string

In [3]:
descriptions = jobs_for_summary["description"].astype(str)

job_descriptions = "\n\n---\n\n".join(description for description in descriptions)

print(f"Summarizing use cases across {len(descriptions)} job postings")
print(job_descriptions[:10000] + "\n\n...")

Summarizing use cases across 11 job postings
**Job Description Summary**


At GE Aerospace, we tackle some of the world's most complex engineering challenges. Our engineers rely on software developers to bridge data, tools, and cutting\-edge technologies like cloud and Al, unlocking new possibilities and advancing the future of flight.  

  

As an intern, you'll build expertise in cloud platforms, databases, data engineering, web design, Al technologies, and numerical methods to solve real\-world engineering problems. Through hands\-on projects, you'll work with the Innovator Method, Game Theory, and DecisionIQ, turning customer insights into impactful solutions. You'll have the opportunity to learn, contribute, and bring real business value and help shape the future of aerospace design.**Job Description**

**What You will Do**

* Develop technical solutions for engineers to evaluate and adopt data and AI solutions.
* Consult on architecture, implement proof of concepts and deliver so

### Prepare the prompt

We combine the job descriptions into one prompt and ask the model to summarize the common AI application use cases.

In [6]:
prompt = f"""
Read the job descriptions below and summarize the AI application use cases that companies are building.

Return markdown in exactly this format:
### Shared use cases
- 5 concise bullet points

Requirements:
- Focus on the applications, workflows, or business use cases being built
- Ignore benefits, recruiting language, company marketing, and hiring process details
- Keep bullets concrete and specific
- Do not mention company names
- Do not restate generic engineering responsibilities

Job descriptions:
{job_descriptions}
""".strip()

print("Prompt for summarization:")
print(prompt[:1000])

Prompt for summarization:
Read the job descriptions below and summarize the AI application use cases that companies are building.

Return markdown in exactly this format:
### Shared use cases
- 5 concise bullet points

Requirements:
- Focus on the applications, workflows, or business use cases being built
- Ignore benefits, recruiting language, company marketing, and hiring process details
- Keep bullets concrete and specific
- Do not mention company names
- Do not restate generic engineering responsibilities

Job descriptions:
**Job Description Summary**


At GE Aerospace, we tackle some of the world's most complex engineering challenges. Our engineers rely on software developers to bridge data, tools, and cutting\-edge technologies like cloud and Al, unlocking new possibilities and advancing the future of flight.  

  

As an intern, you'll build expertise in cloud platforms, databases, data engineering, web design, Al technologies, and numerical methods to solve real\-world engine


### Summarize the use cases

We send the prompt to the model and display the response as markdown.

In [7]:
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-mini",
    input=prompt,
    text={"verbosity": "low"},
)

display(Markdown(response.output_text))

### Shared use cases
- Build enterprise AI agents that use tool calling, multi-step reasoning, and orchestration to automate internal workflows and exception handling.
- Create RAG and knowledge-based systems over private documents, enterprise data, and operational repositories for grounded answers and decision support.
- Develop AI workflows for classification, routing, summarization, text-to-SQL, and data extraction from structured and unstructured business data.
- Ship conversational interfaces and voice-based AI experiences that connect users to agents and backend systems in real time.
- Add evaluation, observability, guardrails, and feedback loops to monitor AI quality, safety, reliability, and continuous improvement in production.